RetailPulse 360

Notebook 02 — Stylo Store & City Network

Goal: Build the physical footprint of Stylo's simulated store network — 190 stores across
94 Pakistani cities/towns spanning every province — then assign each store a real behavioral
"personality" (sampled from Notebook 01's Rossmann-derived profiles) so later notebooks have
a realistic, non-uniform demand pattern to build on.

Input: store_personalities.csv (from Notebook 01)
Output: stores.csv, cities.csv

In [12]:
# 1. IMPORTS
# ============================================================

import pandas as pd
import numpy as np

print("Libraries imported successfully.")

Libraries imported successfully.


In [13]:
# 2. LOAD STORE PERSONALITIES
# ============================================================

PERSONALITIES_PATH = "/kaggle/input/datasets/hamaz911/output/store_personalities.csv"

personalities = pd.read_csv(PERSONALITIES_PATH)

print("Personalities loaded successfully.")
print("Shape:", personalities.shape)
personalities.head()

Personalities loaded successfully.
Shape: (1115, 12)


,rossmann_store_id,dow_mon,dow_tue,dow_wed,dow_thu,dow_fri,dow_sat,dow_sun,promo_lift,trend_pct_per_year,volatility_cv,holiday_lift
0,1,1.088015,0.984562,0.957264,0.936699,0.993147,1.038636,1.0,0.2269,-0.0330,0.2127,0.0099
1,2,1.223673,1.083833,1.177849,1.002740,0.942609,0.579968,1.0,0.6245,0.0288,0.3250,0.0948
2,3,1.201053,1.101155,1.027312,0.999117,1.036101,0.643564,1.0,0.6450,0.0018,0.3159,0.0813
3,4,1.125869,0.976293,0.925463,0.941326,0.982571,1.049287,1.0,0.1923,0.0390,0.2009,0.0641
4,5,1.316537,1.092685,1.114581,1.015330,1.038873,0.444607,1.0,0.7363,0.0172,0.3776,0.0766


In [14]:
# 3. DATA QUALITY — VALIDATE LOADED PERSONALITIES
# ============================================================

print("Row count:", len(personalities))
assert len(personalities) == 1115, "Expected 1115 personality profiles — got a different count"

print("Missing values:")
missing = personalities.isna().sum()
print(missing[missing > 0] if missing.sum() > 0 else "None")

print("\nDuplicate rossmann_store_id:", personalities["rossmann_store_id"].duplicated().sum())

print("\nSanity range check on ratio columns:")
ratio_cols = ["dow_mon","dow_tue","dow_wed","dow_thu","dow_fri","dow_sat","dow_sun",
              "promo_lift","trend_pct_per_year","volatility_cv","holiday_lift"]
print(personalities[ratio_cols].describe().round(3).loc[["min","max"]])

Row count: 1115
Missing values:
None

Duplicate rossmann_store_id: 0

Sanity range check on ratio columns:
     dow_mon  dow_tue  dow_wed  dow_thu  dow_fri  dow_sat  dow_sun  \
min    0.958    0.845    0.835    0.854    0.874    0.323    0.262   
max    1.442    1.298    1.244    1.316    1.234    1.392    1.577   

     promo_lift  trend_pct_per_year  volatility_cv  holiday_lift  
min      -0.070              -0.242          0.123        -0.093  
max       1.449               0.365          0.551         0.498  


In [15]:
# 4. DEFINE STYLO CITY NETWORK
# ============================================================
# No external dataset here — this is our own domain-knowledge design,
# since there's no public list of Stylo's actual store locations.
# Store counts per city reflect realistic retail footprint concentration
# by city tier (Metro / Tier2 / Tier3).

cities_raw = [
    ("Lahore", "Punjab", "Metro", 17), ("Karachi", "Sindh", "Metro", 20),
    ("Islamabad", "Islamabad Capital Territory", "Metro", 7), ("Rawalpindi", "Punjab", "Metro", 7),
    ("Faisalabad", "Punjab", "Metro", 6), ("Multan", "Punjab", "Metro", 6),
    ("Peshawar", "Khyber Pakhtunkhwa", "Metro", 6), ("Quetta", "Balochistan", "Metro", 4),
    ("Gujranwala", "Punjab", "Tier2", 5), ("Sialkot", "Punjab", "Tier2", 4),
    ("Hyderabad", "Sindh", "Tier2", 4), ("Bahawalpur", "Punjab", "Tier2", 3),
    ("Sargodha", "Punjab", "Tier2", 3), ("Sukkur", "Sindh", "Tier2", 2),
    ("Larkana", "Sindh", "Tier2", 2), ("Sheikhupura", "Punjab", "Tier2", 2),
    ("Rahim Yar Khan", "Punjab", "Tier2", 2), ("Jhang", "Punjab", "Tier2", 2),
    ("Dera Ghazi Khan", "Punjab", "Tier2", 2), ("Gujrat", "Punjab", "Tier2", 2),
    ("Sahiwal", "Punjab", "Tier2", 2), ("Wah Cantt", "Punjab", "Tier2", 2),
    ("Mardan", "Khyber Pakhtunkhwa", "Tier2", 2), ("Abbottabad", "Khyber Pakhtunkhwa", "Tier2", 2),
    ("Mingora", "Khyber Pakhtunkhwa", "Tier2", 2), ("Kasur", "Punjab", "Tier3", 2),
    ("Okara", "Punjab", "Tier3", 2), ("Chiniot", "Punjab", "Tier3", 1),
    ("Kamoke", "Punjab", "Tier3", 1), ("Muzaffargarh", "Punjab", "Tier3", 1),
    ("Khanewal", "Punjab", "Tier3", 1), ("Vehari", "Punjab", "Tier3", 1),
    ("Jhelum", "Punjab", "Tier3", 2), ("Mianwali", "Punjab", "Tier3", 1),
    ("Attock", "Punjab", "Tier3", 1), ("Kot Addu", "Punjab", "Tier3", 1),
    ("Toba Tek Singh", "Punjab", "Tier3", 1), ("Hafizabad", "Punjab", "Tier3", 1),
    ("Nowshera", "Khyber Pakhtunkhwa", "Tier3", 1), ("Kohat", "Khyber Pakhtunkhwa", "Tier3", 1),
    ("Dera Ismail Khan", "Khyber Pakhtunkhwa", "Tier3", 1), ("Swabi", "Khyber Pakhtunkhwa", "Tier3", 1),
    ("Charsadda", "Khyber Pakhtunkhwa", "Tier3", 1), ("Nawabshah", "Sindh", "Tier3", 2),
    ("Mirpur Khas", "Sindh", "Tier3", 1), ("Jacobabad", "Sindh", "Tier3", 1),
    ("Shikarpur", "Sindh", "Tier3", 1), ("Khairpur", "Sindh", "Tier3", 1),
    ("Dadu", "Sindh", "Tier3", 1), ("Badin", "Sindh", "Tier3", 1),
    ("Thatta", "Sindh", "Tier3", 1), ("Turbat", "Balochistan", "Tier3", 1),
    ("Khuzdar", "Balochistan", "Tier3", 1), ("Chaman", "Balochistan", "Tier3", 1),
    ("Gwadar", "Balochistan", "Tier3", 1), ("Zhob", "Balochistan", "Tier3", 1),
    ("Muzaffarabad", "Azad Kashmir", "Tier2", 2), ("Mirpur (AJK)", "Azad Kashmir", "Tier3", 1),
    ("Gilgit", "Gilgit-Baltistan", "Tier3", 1), ("Skardu", "Gilgit-Baltistan", "Tier3", 1),
    ("Chakwal", "Punjab", "Tier3", 1), ("Narowal", "Punjab", "Tier3", 1),
    ("Pakpattan", "Punjab", "Tier3", 1), ("Layyah", "Punjab", "Tier3", 1),
    ("Bhakkar", "Punjab", "Tier3", 1), ("Jaranwala", "Punjab", "Tier3", 1),
    ("Daska", "Punjab", "Tier3", 1), ("Pattoki", "Punjab", "Tier3", 1),
    ("Mandi Bahauddin", "Punjab", "Tier3", 1), ("Khushab", "Punjab", "Tier3", 1),
    ("Ferozewala", "Punjab", "Tier3", 1), ("Wazirabad", "Punjab", "Tier3", 1),
    ("Kabirwala", "Punjab", "Tier3", 1), ("Lodhran", "Punjab", "Tier3", 1),
    ("Ahmedpur East", "Punjab", "Tier3", 1), ("Chishtian", "Punjab", "Tier3", 1),
    ("Hasilpur", "Punjab", "Tier3", 1), ("Haroonabad", "Punjab", "Tier3", 1),
    ("Mianchannu", "Punjab", "Tier3", 1), ("Shorkot", "Punjab", "Tier3", 1),
    ("Ghotki", "Sindh", "Tier3", 1), ("Tando Adam", "Sindh", "Tier3", 1),
    ("Tando Allahyar", "Sindh", "Tier3", 1), ("Umerkot", "Sindh", "Tier3", 1),
    ("Naushahro Feroze", "Sindh", "Tier3", 1), ("Kandhkot", "Sindh", "Tier3", 1),
    ("Haripur", "Khyber Pakhtunkhwa", "Tier3", 1), ("Bannu", "Khyber Pakhtunkhwa", "Tier3", 1),
    ("Batkhela", "Khyber Pakhtunkhwa", "Tier3", 1), ("Lakki Marwat", "Khyber Pakhtunkhwa", "Tier3", 1),
    ("Tank", "Khyber Pakhtunkhwa", "Tier3", 1), ("Loralai", "Balochistan", "Tier3", 1),
    ("Sibi", "Balochistan", "Tier3", 1), ("Kalat", "Balochistan", "Tier3", 1),
]

cities_df = pd.DataFrame(cities_raw, columns=["city", "province", "city_tier", "n_stores"])

print("Cities defined:", len(cities_df))
print("Total stores across cities:", cities_df["n_stores"].sum())
cities_df.head()

Cities defined: 94
Total stores across cities: 190


,city,province,city_tier,n_stores
0,Lahore,Punjab,Metro,17
1,Karachi,Sindh,Metro,20
2,Islamabad,Islamabad Capital Territory,Metro,7
3,Rawalpindi,Punjab,Metro,7
4,Faisalabad,Punjab,Metro,6


In [16]:
# 5. DATA QUALITY — VALIDATE CITY TABLE
# ============================================================

print("Total cities:", len(cities_df))
assert len(cities_df) == cities_df["city"].nunique(), "Duplicate city names found"

print("Total stores across all cities:", cities_df["n_stores"].sum())
assert cities_df["n_stores"].sum() == 190, "Store total doesn't match expected 190"

print("\nStores by province:")
print(cities_df.groupby("province")["n_stores"].sum().sort_values(ascending=False))

print("\nStores by city tier:")
print(cities_df.groupby("city_tier")["n_stores"].sum())

print("\nAny zero or negative store counts?", (cities_df["n_stores"] <= 0).sum())

Total cities: 94
Total stores across all cities: 190

Stores by province:
province
Punjab                         101
Sindh                           43
Khyber Pakhtunkhwa              22
Balochistan                     12
Islamabad Capital Territory      7
Azad Kashmir                     3
Gilgit-Baltistan                 2
Name: n_stores, dtype: int64

Stores by city tier:
city_tier
Metro    73
Tier2    45
Tier3    72
Name: n_stores, dtype: int64

Any zero or negative store counts? 0


In [17]:
# 6. EXPAND CITIES INTO INDIVIDUAL STORES
# ============================================================
# Store size tier (Large/Medium/Small) is probabilistic, weighted by
# city tier — metros skew toward bigger stores, small towns skew small.
# Seeded for reproducibility.

np.random.seed(42)

rows = []
store_counter = 1

for _, r in cities_df.iterrows():
    for i in range(int(r["n_stores"])):
        if r["city_tier"] == "Metro":
            size = np.random.choice(["Large", "Medium", "Small"], p=[0.35, 0.45, 0.20])
        elif r["city_tier"] == "Tier2":
            size = np.random.choice(["Large", "Medium", "Small"], p=[0.10, 0.45, 0.45])
        else:  # Tier3
            size = np.random.choice(["Medium", "Small"], p=[0.25, 0.75])

        rows.append({
            "store_id": f"STY-{store_counter:04d}",
            "city": r["city"],
            "province": r["province"],
            "city_tier": r["city_tier"],
            "store_size": size,
        })
        store_counter += 1

stores_df = pd.DataFrame(rows)

print("Stores created:", len(stores_df))
print("\nStore size distribution:")
print(stores_df["store_size"].value_counts())
stores_df.head()

Stores created: 190

Store size distribution:
store_size
Small     88
Medium    65
Large     37
Name: count, dtype: int64


,store_id,city,province,city_tier,store_size
0,STY-0001,Lahore,Punjab,Metro,Medium
1,STY-0002,Lahore,Punjab,Metro,Small
2,STY-0003,Lahore,Punjab,Metro,Medium
3,STY-0004,Lahore,Punjab,Metro,Medium
4,STY-0005,Lahore,Punjab,Metro,Large


In [18]:
# 7. ADD NATIONAL ONLINE CHANNEL
# ============================================================
# Stylo has some e-commerce presence alongside physical stores.
# We represent this as one national "virtual store" entry rather
# than trying to invent per-city online breakdowns we have no basis for.

stores_df["channel"] = "Physical"

ecom_row = pd.DataFrame([{
    "store_id": "STY-ECOM",
    "city": "National (Online)",
    "province": "National",
    "city_tier": "Online",
    "store_size": "N/A",
    "channel": "Online",
}])

stores_df = pd.concat([stores_df, ecom_row], ignore_index=True)

print("Total rows now:", len(stores_df))
print(stores_df["channel"].value_counts())
stores_df.tail(3)

Total rows now: 191
channel
Physical    190
Online        1
Name: count, dtype: int64


,store_id,city,province,city_tier,store_size,channel
188,STY-0189,Sibi,Balochistan,Tier3,Small,Physical
189,STY-0190,Kalat,Balochistan,Tier3,Medium,Physical
190,STY-ECOM,National (Online),National,Online,N/A,Online


In [19]:
# 8. ASSIGN ROSSMANN PERSONALITIES TO PHYSICAL STORES
# ============================================================
# Sampled WITHOUT replacement (190 physical stores < 1115 available
# personalities, so no need to reuse any). Random and independent of
# store_size/city_tier — we have no real evidence that, say, "Large
# stores are more promo-sensitive," so we don't fabricate that
# correlation just because it would look tidy.
#
# STY-ECOM (online channel) is deliberately excluded — Rossmann's
# personality traits describe physical foot-traffic behavior, which
# doesn't meaningfully apply to an online channel.

np.random.seed(7)

physical_mask = stores_df["channel"] == "Physical"
n_physical = physical_mask.sum()

sampled_personalities = personalities.sample(
    n=n_physical, replace=False, random_state=7
).reset_index(drop=True)

# attach the sampled personalities row-by-row to physical stores only
physical_stores = stores_df[physical_mask].reset_index(drop=True)
physical_stores = pd.concat([physical_stores, sampled_personalities], axis=1)

# the online store keeps no personality columns (will be NaN when merged back)
online_store = stores_df[~physical_mask].copy()

stores_df = pd.concat([physical_stores, online_store], ignore_index=True)

print("Final stores_df shape:", stores_df.shape)
print("Physical stores with personality assigned:", stores_df["rossmann_store_id"].notna().sum())
print("Online rows (expected no personality):", stores_df["rossmann_store_id"].isna().sum())
stores_df.head()

Final stores_df shape: (191, 18)
Physical stores with personality assigned: 190
Online rows (expected no personality): 1


,store_id,city,province,city_tier,store_size,channel,rossmann_store_id,dow_mon,dow_tue,dow_wed,dow_thu,dow_fri,dow_sat,dow_sun,promo_lift,trend_pct_per_year,volatility_cv,holiday_lift
0,STY-0001,Lahore,Punjab,Metro,Medium,Physical,47.0,1.154211,1.048033,1.047515,0.997544,0.972048,0.787093,1.0,0.4084,0.0685,0.2443,0.0161
1,STY-0002,Lahore,Punjab,Metro,Small,Physical,450.0,1.044594,0.906838,0.905348,1.080835,0.931722,1.134867,1.0,0.2520,0.0176,0.2456,0.0544
2,STY-0003,Lahore,Punjab,Metro,Medium,Physical,180.0,1.133773,0.974817,0.955613,0.969955,0.967307,1.001073,1.0,0.3071,0.0238,0.2161,-0.0153
3,STY-0004,Lahore,Punjab,Metro,Medium,Physical,721.0,1.043749,0.943569,0.946225,0.968320,1.074893,1.025539,1.0,0.1783,0.0373,0.1917,-0.0218
4,STY-0005,Lahore,Punjab,Metro,Large,Physical,320.0,1.184150,1.044030,0.948902,1.005677,0.972349,0.851640,1.0,0.3061,0.0317,0.2299,0.0325


In [20]:
# 9. DATA QUALITY — VALIDATE FINAL STORE TABLE
# ============================================================

print("Total rows:", len(stores_df))
assert len(stores_df) == 191, "Expected 191 rows (190 physical + 1 online)"

print("Duplicate store_id:", stores_df["store_id"].duplicated().sum())
assert stores_df["store_id"].duplicated().sum() == 0, "Duplicate store_id found"

print("Duplicate rossmann_store_id (personality reuse):",
      stores_df["rossmann_store_id"].dropna().duplicated().sum())
assert stores_df["rossmann_store_id"].dropna().duplicated().sum() == 0, "A personality was assigned twice"

physical = stores_df[stores_df["channel"] == "Physical"]
print("\nPhysical stores missing a personality:", physical["rossmann_store_id"].isna().sum())
assert physical["rossmann_store_id"].isna().sum() == 0, "Some physical store has no personality assigned"

online = stores_df[stores_df["channel"] == "Online"]
print("Online row correctly has no personality:", online["rossmann_store_id"].isna().all())

print("\nFinal store size distribution:")
print(stores_df["store_size"].value_counts())

Total rows: 191
Duplicate store_id: 0
Duplicate rossmann_store_id (personality reuse): 0

Physical stores missing a personality: 0
Online row correctly has no personality: True

Final store size distribution:
store_size
Small     88
Medium    65
Large     37
N/A        1
Name: count, dtype: int64


In [21]:
# 9b. FIX — RENAME PROVINCE TO REGION (ACCURATE LABELING)
# ============================================================
# Pakistan officially has 4 provinces (Punjab, Sindh, KPK, Balochistan)
# + Islamabad Capital Territory as a federal territory — 5 official
# administrative units. Azad Kashmir and Gilgit-Baltistan are
# self-governing territories, not provinces. Labeling all of these
# as "province" was incorrect. Renaming the column to "region" to
# accurately reflect that it holds a mix of provinces, one federal
# territory, and two autonomous territories.

cities_df = cities_df.rename(columns={"province": "region"})
stores_df = stores_df.rename(columns={"province": "region"})

print("Distinct regions:", stores_df["region"].nunique())
print(stores_df["region"].value_counts())

Distinct regions: 8
region
Punjab                         101
Sindh                           43
Khyber Pakhtunkhwa              22
Balochistan                     12
Islamabad Capital Territory      7
Azad Kashmir                     3
Gilgit-Baltistan                 2
National                         1
Name: count, dtype: int64


In [22]:
# 10. SAVE OUTPUTS
# ============================================================

stores_df.to_csv("stores.csv", index=False)
cities_df.to_csv("cities.csv", index=False)

print("Saved stores.csv —", stores_df.shape[0], "rows,", stores_df.shape[1], "columns")
print("Saved cities.csv —", cities_df.shape[0], "rows,", cities_df.shape[1], "columns")

Saved stores.csv — 191 rows, 18 columns
Saved cities.csv — 94 rows, 4 columns


In [24]:
# 11. NOTEBOOK SUMMARY
# ============================================================

print("NOTEBOOK 02 — STYLO STORE & CITY NETWORK")
print(f"Cities/towns: {len(cities_df)}")
print(f"Physical stores: {(stores_df['channel']=='Physical').sum()}")
print(f"Online channel entries: {(stores_df['channel']=='Online').sum()}")
print(f"Regions covered: {stores_df['region'].nunique()}")
print(f"Store sizes: {dict(stores_df['store_size'].value_counts())}")
print(f"Personalities assigned from Rossmann pool: {stores_df['rossmann_store_id'].notna().sum()} / 1115 available")
print()
print("Output files: stores.csv, cities.csv")
print()
print("✓ Notebook 02 completed successfully.")

NOTEBOOK 02 — STYLO STORE & CITY NETWORK
Cities/towns: 94
Physical stores: 190
Online channel entries: 1
Regions covered: 8
Store sizes: {np.str_('Small'): np.int64(88), np.str_('Medium'): np.int64(65), np.str_('Large'): np.int64(37), 'N/A': np.int64(1)}
Personalities assigned from Rossmann pool: 190 / 1115 available

Output files: stores.csv, cities.csv

✓ Notebook 02 completed successfully.


In [25]:
# 12. ZIP ARTIFACTS FOR LOCAL DOWNLOAD
# ============================================================
import shutil
from pathlib import Path

OUTPUT_FILES = [Path("stores.csv"), Path("cities.csv")]
ZIP_NAME = "notebook_02_store_city_network_artifact"
ARTIFACT_DIR = Path("notebook_02_outputs")

missing = [f for f in OUTPUT_FILES if not f.exists()]
if missing:
    raise FileNotFoundError(
        f"Missing output file(s): {missing}. Run the save cell (Section 10) first."
    )

# collect both files into one folder, then zip the folder
ARTIFACT_DIR.mkdir(exist_ok=True)
for f in OUTPUT_FILES:
    shutil.copy(f, ARTIFACT_DIR / f.name)

zip_path = shutil.make_archive(ZIP_NAME, "zip", root_dir=".", base_dir=ARTIFACT_DIR.name)

print("ZIP created successfully:")
print(zip_path)
print(f"ZIP size: {Path(zip_path).stat().st_size / 1024:.2f} KB")
print(f"Contents: {[f.name for f in OUTPUT_FILES]}")

ZIP created successfully:
/kaggle/working/notebook_02_store_city_network_artifact.zip
ZIP size: 16.56 KB
Contents: ['stores.csv', 'cities.csv']
